## CT25_10_TG_Parquet_to_TG_CT
<b>Env</b>: cdms_plus (Python 3.12.7)


#### Major steps required
- Read in CDMS Parquet data
- Produce Cyclic Top sinewave
- Produce Cyclic Top peak amplitude data
- Partition and write the parquet track geometry data to the drive
- Partition and write the parquet sinewave data to the drive
- Partition and write the parquet peak amplitude data to the drive

<br>
<hr style="border:1px solid blue">

#### Imports

In [1]:
from scipy.signal import sosfilt
import numpy as np
import pyarrow.parquet as pq
import pandas as pd
import datetime as dt
from os import walk
import time
from datetime import datetime
from pathlib import Path

pd.options.display.max_columns=30
start_time = time.time()

#### Initialisation and settings

In [ ]:
proj_root = 'C:/Users/s2887048/Documents/network_rail/CPRICS/data_acquired_oct_26/@Adnan/'

data_in_path = proj_root + 'Parquet/'                           # CDMS --> parquet files

data_out_tg = proj_root + 'Data_Out_TG/'                        # Where to place the new track geometry parquet files
data_out_sine =  proj_root + 'Data_Out_Sinewave/'               # Where to place the new CT sinewave parquet files
data_out_peak_amp =  proj_root + 'Data_Out_Peak_Amp/'           # Where to place the new CT peak amplitude parquet files
data_out_peak_amp_CSV =  proj_root + 'Data_Out_Peak_Amp_CSV/'   # Used for debugging

data_out_tg_ref = proj_root + 'Data_Out_TG_Refs/'               # Partition the output date by filename ('Ref'), ELR, TrackID
data_out_peak_amp_ref =  proj_root + 'Data_Out_Peak_Amp_Refs/'  # Partition the output date by filename ('Ref'), ELR, TrackID
data_out_sine_ref = proj_root + 'Data_Out_Sinewave_Refs/'       # Partition the output date by filename ('Ref'), ELR, TrackID

risk_sites = proj_root + 'CT_Risk_Sites/'                       # Listing of three or more peak amplitudes above the threshold (per Ref)
risk_sites_grouped = proj_root + 'CT_Risk_Sites_Grouped/'       # Listing of three or more peak amplitudes above the threshold, grouped and categorised (per Ref)
charts = proj_root + 'Charts/'                                  # Only used for analysis of output, not the processing


engine = 'pyarrow'                                  # Parquet engine to use
comp = 'snappy'                                     # Compression format
tg_part_cols = ['ELR', 'TrackID']                   # How to partition the parquet files
ct_part_cols = ['ELR', 'TrackID']                   # How to partition the parquet files - Normally
ct_part_cols_filename = ['Ref', 'ELR', 'TrackID']   # Partition the output date by filename ('Ref'), ELR, TrackID

METRES_TO_YARDS = 1.0936133

ANH_DEBUG = False                   # Debug - ie, output CSVs etc
BY_FILENAME = False                 # Partition the output date by filename ('Ref'), ELR, TrackID

filter_type = 'Elliptic'            # Bandpass 'filter_type' - Just in case other filters are needed 
APPLY_PHASE_COMPENSATION = True     # Apply Phase Compensation - corrects each wavelength for Group Delay

In [3]:
# This def creates the sub-directories in 'dirs_to_create' if they don't exist
def make_dirs(dirs):
    for d in dirs:
        Path(d).mkdir(parents=True, exist_ok=True)

In [4]:
# Directories needed for the process - Create if they don't exist

if BY_FILENAME:

    dirs_to_create = [
        data_in_path,
        data_out_tg_ref,
        data_out_sine_ref,
        data_out_peak_amp_ref,
        data_out_peak_amp_CSV,
        risk_sites,
        risk_sites_grouped,
        charts
        ]

else:
    dirs_to_create = [
        data_in_path,
        data_out_tg,
        data_out_sine,
        data_out_peak_amp,
        data_out_peak_amp_CSV,
        risk_sites,
        risk_sites_grouped,
        charts
        ]


make_dirs(dirs_to_create)


***
### General Defs

In [5]:
# Functions which takes filename and outputs date

def MultiDate(datedata):
      
    """
    Handles different potential input formats of dates

    Parameters: datedata - string type, date characters extracted from filename

    Returns: a string type, formatted date in correct character order

    Example:
        MultiDate('060718')   ----> '2018-07-06'
        MultiDate('20200706') ----> '2020-07-06'
    """

    # Convert the date strings into list of Python date formats
    if len(datedata)==8:
        strdate = str(dt.datetime.strptime(datedata, "%Y%m%d").date())

    elif len(datedata)==6:
        strdate = str(dt.datetime.strptime(datedata, "%d%m%y").date())

    else:
        raise ValueError('Date not in correct format')

    return strdate


def ExtractDate(file_name):
    
    if file_name.startswith("TrackSysExport"): 
        splits = 2                                          # How many bits the data is split into
    else:
        splits = 4

    split_parquet_file_name = file_name.split('_',splits)   # Split parquet_file by '_' (five parts)
    date = split_parquet_file_name[splits-1]                # Get the string date part
    sinewave_date = MultiDate(date)                         # Convert to a date
    return sinewave_date


#### Defs for CT

In [6]:
def prepare_geometry_signal(series):

    numeric_values = pd.to_numeric(
        series,
        errors='coerce'
    )

    return numeric_values.interpolate(
        method='linear',
        limit_direction='both'
    )

In [7]:
def create_track_filtered(Cyclic_Tops, filter_type):

    if filter_type == 'Elliptic':
        filter_function = ellip_func          # Use the function described below. In section titled 'Bandpass Filter Design and Parameters
    else:
        raise ValueError(
            f"Unknown filter_type: {filter_type}"
        )

    track_column_map = {
        'Top_Right35': 'top_right',
        'Top_Left35': 'top_left'
    }

    wavelength_column_map = {
        4.5: '45',
        6.0: '6',
        9.0: '9',
        13.5: '135',
        18.0: '18'
    }


    CT_DELAY_SAMPLES = {
        4.5: 24,
        6.0: 32,
        9.0: 47,
        13.5: 68,
        18.0: 96
    }


    sinewave_df = Cyclic_Tops.copy()

    for geometry_column, output_suffix in track_column_map.items():

        geometry_values = (
            prepare_geometry_signal(
                sinewave_df[geometry_column]
            )
            .to_numpy(dtype=float)
        )


        for wavelength, wavelength_label in wavelength_column_map.items():

            output_column = (
                f'{wavelength_label}_{output_suffix}'
            )

            filtered_signal = filter_function(
                wavelength,
                geometry_values
            )

            if APPLY_PHASE_COMPENSATION:
                delay_samples = CT_DELAY_SAMPLES[wavelength]    # For each wavelength compensate for delay
            else:
                delay_samples = 0                               # Don't compensate


            phase_compensated_signal = (
                pd.Series(filtered_signal)
                .shift(-delay_samples)
            )

            sinewave_df[output_column] = phase_compensated_signal


    return sinewave_df

In [8]:
def find_max_peaks(All_Data):
      
    # re-order the columns
    data2 = All_Data[['CT_Output_Index', 'distance', 'ELR', 'TrackID', 'Ref', 'Location_m', 
                    'Top_Right35', 'Top_Left35', '45_top_right', '45_top_left', 
                    '6_top_right', '6_top_left', '9_top_right', '9_top_left', '135_top_right', '135_top_left', 
                    '18_top_right', '18_top_left']]
     

    # Metres -> yards
    data2['Location_Y'] = data2['Location_m'] * METRES_TO_YARDS

    # Miles and remaining yards
    miles = (data2['Location_Y'] // 1760).astype(int)

    remaining_yards = (
        data2['Location_Y'] - (miles * 1760)
    ).astype(int)

    # Mileage format
    data2['Mileage'] = np.where(
        data2['Location_m'] < 0,
        np.round(data2['Location_m'], 0),
        miles + remaining_yards / 10000
    )

  
    # re-order and rename the columns
    data_reordered = data2[['CT_Output_Index', 'distance', 'ELR', 'TrackID', 'Ref', 'Location_Y', 'Location_m', 'Mileage', 'Top_Right35', 
                            'Top_Left35', '45_top_right', '45_top_left', '6_top_right', '6_top_left', '9_top_right', 
                            '9_top_left', '135_top_right', '135_top_left', '18_top_right', '18_top_left']]

    data_renamed = data_reordered.rename(columns={"45_top_right": "45TR", "45_top_left": "45TL", "6_top_right": "6TR", "6_top_left": "6TL", "9_top_right": "9TR", "9_top_left": "9TL", "135_top_right": "135TR", "135_top_left": "135TL", "18_top_right": "18TR", "18_top_left": "18TL", "top_right": "Top_Right", "top_left": "Top_Left"})
  
    # replace all negative values with 0
    cols = ['45TR', '45TL', '6TR', '6TL', '9TR', '9TL', '135TR', '135TL', '18TR', '18TL']
    for c in cols:
        data_renamed[c] = data_renamed[c].clip(lower=0)
    

    # find the maximum value for each array of positive values
    def find_peak(col):

        arr = np.asarray(col)

        peaks = np.where(
            (arr >= np.r_[0, arr[:-1]]) &
            (arr >= np.r_[arr[1:], 0]),
            arr,
            0
        )

        return peaks
    

    for c in cols:
        data_renamed[c] = find_peak(data_renamed[c])

    data_renamed[cols] = (data_renamed[cols].replace(0, np.nan))
 
    # re-order and re-name the output columns
    output = data_renamed[['CT_Output_Index', 'ELR', 'TrackID', 'Mileage', 'Top_Right35', 'Top_Left35', 
                           '45TR', '45TL', '6TR', '6TL', '9TR', '9TL', '135TR', '135TL', '18TR', '18TL',
                           'distance', 'Ref', 'Location_m', 'Location_Y']] 

    output = output.rename(columns={"45TR": "45_top_right", "45TL": "45_top_left", "6TR": "6_top_right", "6TL": "6_top_left",
                                    "9TR": "9_top_right", "9TL": "9_top_left", "135TR": "135_top_right", 
                                    "135TL": "135_top_left", "18TR": "18_top_right", "18TL": "18_top_left", 
                                    "distance": "Distance"})
  
    return output

In [9]:
# # add distance and save output from first step

def generate_all_data_peaks(Tops):
  
    # Add in distance col which increases by 0.2 each row
    Tops['distance'] = np.round(
    np.arange(1, len(Tops) + 1) * 0.2,1)
   
    return Tops

In [10]:
def save_file_All_Data(s_data, fdate):  
     
    s_data = s_data[['CT_Output_Index', 'ELR', 'TrackID', 'Mileage', 'Top_Right35', 'Top_Left35', '45_top_right', '45_top_left',
                 '6_top_right', '6_top_left', '9_top_right', '9_top_left', '135_top_right', '135_top_left', '18_top_right', 
                 '18_top_left', 'Distance', 'Ref', 'Location_m', 'Location_Y']] 

    s_data = s_data.rename(columns={"CT_Output_Index": "CT_Index"})
      
    s_data['Date']=fdate

    #------------------------------------------------------------------------------------------------------------------------------
    # Output the partitioned parquet file by 'Ref', 'ELR', and 'TrackID'
    # Just in case a Risk Site starts in one ELR/TrackID and ends in another
    # Or just be 'ELR' and 'TrackID'
    if BY_FILENAME:
        s_data.to_parquet(
            data_out_peak_amp_ref,
            partition_cols=ct_part_cols_filename, 
            engine=engine, 
            compression=comp, 
            index=False
            )
    else:
        s_data.to_parquet(
            data_out_peak_amp,
            partition_cols=ct_part_cols, 
            engine=engine, 
            compression=comp, 
            index=False
            )

    #-------------------------------------------------------------------------------------------------------------------------------

    if ANH_DEBUG:
        filename = s_data['Ref'].iloc[0]                            # Get the filename from the first row
        s_data.to_csv(data_out_peak_amp_CSV + filename + '.csv')


In [11]:
def runfiles(Cyclic_Tops, fdate):

    sinewave = create_track_filtered(
        Cyclic_Tops=Cyclic_Tops,
        filter_type=filter_type
    )

    sinewave['Date'] = fdate


    #------------------------------------------------------------------------------------------------------------------------------
    if BY_FILENAME:
        sinewave.to_parquet(
            data_out_sine_ref,
            partition_cols=ct_part_cols_filename,
            engine=engine,
            compression=comp,
            index=False
        )

    else:

        sinewave.to_parquet(
            data_out_sine,
            partition_cols=ct_part_cols,
            engine=engine,
            compression=comp,
            index=False
        )
    #------------------------------------------------------------------------------------------------------------------------------

    return sinewave

<hr style="border:1px solid blue">

### Bandpass Filter Design and Parameters

---

To implement this Cyclic Top filter design against track geometry measurement data, the following information is given:

<ul>
    <li>The target Cyclic Top wavelengths.</li>
    <li>The track geometry sampling interval.
        <ul>
            <li>Input data should be sampled at 0.20 m intervals (5 samples per metre).</li>
        </ul>
    </li>
    <li>The geometry channels to be filtered:
        <ul>
            <li>Top_Left35</li>
            <li>Top_Right35</li>
        </ul>
    </li>
    <li>The Second Order Section (SOS) filter <code>CT_SOS_FILTERS</code> coefficients.</li>
        <ul>
            <li>The Cyclic Top filters should be implemented using the supplied SOS coefficients</li>
            <li>The coefficients are the definitive specification of the filter</li>
        </ul>
    <li>All Cyclic Top filters are implemented as sixth-order elliptic band-pass filters realised as three two pole cascaded second-order sections (SOS).</li>
    <li>Filtering to be performed using a single-pass causal IIR implementation equivalent to <code>scipy.signal.sosfilt()</code>.</li>
    <li>Filtering should be applied sequentially in the order in which the track geometry data was recorded.</li>
    <li>A wavelength-dependent delay correction should be applied following filtering to compensate for filter phase delay.</li>
    <li>The filter will start with zero initial conditions.</li>
    <li>Input and output units shall be millimetres (mm).</li>
</ul>


<h4>Cyclic Top Wavelength Bandpass Ranges</h4>

<table>
    <thead>
        <tr>
            <th style="text-align:right;">Wavelength (m)</th>
            <th style="text-align:right;">Bandpass Lower (m)</th>
            <th style="text-align:right;">Bandpass Upper (m)</th>
            <th style="text-align:right;">Phase Delay Compensation</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="text-align:right;">4.5</td>
            <td style="text-align:right;">3.82</td>
            <td style="text-align:right;">5.48</td>
            <td style="text-align:right;">24</td>
        </tr>
        <tr>
            <td style="text-align:right;">6.0</td>
            <td style="text-align:right;">5.10</td>
            <td style="text-align:right;">7.30</td>
            <td style="text-align:right;">32</td>
        </tr>
        <tr>
            <td style="text-align:right;">9.0</td>
            <td style="text-align:right;">7.64</td>
            <td style="text-align:right;">10.96</td>
            <td style="text-align:right;">47</td>
        </tr>
        <tr>
            <td style="text-align:right;">13.5</td>
            <td style="text-align:right;">11.03</td>
            <td style="text-align:right;">15.83</td>
            <td style="text-align:right;">68</td>
        </tr>
        <tr>
            <td style="text-align:right;">18.0</td>
            <td style="text-align:right;">15.70</td>
            <td style="text-align:right;">22.52</td>
            <td style="text-align:right;">96</td>
        </tr>
    </tbody>
</table>


The causal filter introduces 'phase delay', this is the positional offset introduced by the causal filter.<br>
Cyclic Top features in the filtered output appear slightly later than their true location in the original track geometry record.<br>
A wavelength-dependent delay correction is therefore applied after filtering to restore the correct track position, this being the 'Phase Delay Compensation'


### Second Order Filter Stages (CT_SOS_FILTERS)

| Wavelength (m) | SOS Stage | b0 | b1 | b2 | a0 | a1 | a2 |
|---:|---:|---:|---:|---:|---:|---:|---:|
| 4.5 | 1 | 0.00172428332937209 | 0 | -0.00172428332937209 | 1 | -1.85809016049517 | 0.927342091322867 |
| 4.5 | 2 | 1 | -1.69886739638698 | 1 | 1 | -1.86239443336835 | 0.959379530535870 |
| 4.5 | 3 | 1 | -1.98362346258090 | 1 | 1 | -1.91890944220186 | 0.970200664445812 |
| 6.0 | 1 | 0.00129150270881303 | 0 | -0.00129150270881303 | 1 | -1.90560743413223 | 0.945016077166093 |
| 6.0 | 2 | 1 | -1.82688784817345 | 1 | 1 | -1.91427699416418 | 0.969314529210485 |
| 6.0 | 3 | 1 | -1.99075951888600 | 1 | 1 | -1.94857926091597 | 0.977597110151174 |
| 9.0 | 1 | 0.000864517787457945 | 0 | -0.000864517787457944 | 1 | -1.94530119775838 | 0.963009378001810 |
| 9.0 | 2 | 1 | -1.92183759116850 | 1 | 1 | -1.95475416294721 | 0.979406817699640 |
| 9.0 | 3 | 1 | -1.99588401033950 | 1 | 1 | -1.97205934704101 | 0.985023175397773 |
| 13.5 | 1 | 0.000601690496895342 | 0 | -0.000601690496895343 | 1 | -1.96570359980937 | 0.974245805056640 |
| 13.5 | 2 | 1 | -1.96228818663857 | 1 | 1 | -1.97382232581012 | 0.985688869909266 |
| 13.5 | 3 | 1 | -1.99802542829005 | 1 | 1 | -1.98338013129517 | 0.989611647070966 |
| 18.0 | 1 | 0.000424824258777779 | 0 | -0.000424824258777780 | 1 | -1.97759760171322 | 0.981833326403068 |
| 18.0 | 2 | 1 | -1.98132035929333 | 1 | 1 | -1.98404399342041 | 0.989919133203836 |
| 18.0 | 3 | 1 | -1.99902455545656 | 1 | 1 | -1.98960746639998 | 0.992690152113945 |

*Each Cyclic Top filter is realised as three cascaded Second Order Sections (SOS), forming an overall sixth-order band-pass filter.*

<br>

#### SOS Coefficients

In [12]:
# Second Order Filter Stages

CT_SOS_FILTERS = {
    4.5: [[0.00172428332937209,0,-0.00172428332937209,1,-1.85809016049517, 0.927342091322867],
          [1,-1.69886739638698,1,1,-1.86239443336835, 0.959379530535870],
          [1,-1.98362346258090,1,1,-1.91890944220186, 0.970200664445812]],

    6: [[0.00129150270881303,0,-0.00129150270881303,1,-1.90560743413223,0.945016077166093],
        [1, -1.82688784817345, 1, 1, -1.91427699416418, 0.969314529210485],
        [1, -1.99075951888600, 1, 1, -1.94857926091597, 0.977597110151174]],

    9: [[0.000864517787457945,0,-0.000864517787457944,1,-1.94530119775838,0.963009378001810],
        [1, -1.92183759116850, 1, 1, -1.95475416294721, 0.979406817699640],
        [1, -1.99588401033950, 1, 1, -1.97205934704101, 0.985023175397773]],

    13.5:[[0.000601690496895342,0,-0.000601690496895343,1,-1.96570359980937, 0.974245805056640],
          [1,-1.96228818663857,1,1,-1.97382232581012, 0.985688869909266],
          [1,-1.99802542829005,1,1,-1.98338013129517,0.989611647070966]],

    18: [[0.000424824258777779, 0, -0.000424824258777780, 1, -1.97759760171322, 0.981833326403068],
         [1, -1.98132035929333, 1, 1, -1.98404399342041, 0.989919133203836],
         [1, -1.99902455545656, 1, 1, -1.98960746639998, 0.992690152113945]]
    }


In [13]:
# 6th-order elliptic filter

# Raw Top signal
#       ↓
# 2-pole stage 1
#       ↓
# 2-pole stage 2
#       ↓
# 2-pole stage 3
#       ↓
# Filtered CT signal


def ellip_func(ct_wavelength, geometry):    # Run causal filter

    sos = np.asarray(                       # Convert the list structure (CT_SOS_FILTERS) into a numpy array
        CT_SOS_FILTERS[ct_wavelength],      # Look up the filter coefficients for the wavelength
        dtype=float
    )

    filtered_output = sosfilt(sos, geometry)

    return filtered_output


<hr style="border:1px solid blue">

### Read the data

Read in the parquet file (individual track geometry files)<br>
Output a partitioned track geometry parquet file <br>
Generate the CT sinewave <br>
Output partitioned CT sinewave <br>
Generate the CT peak amplitude <br>
Output partitioned CT peak amplitude <br>
<br>

- Read in a list of the files in the directory
- Iterate through the list of files
    - Read parquet data
    - Add 3 new columns (two of these are used in the partitioning)
        - Id (this is just a sequential index used later to tie together TG, CT_Sinwave, and CT_Peak_Amplitude)
        - Date (this is parsed from the filename)
        - Filename
    - Write the partitioned parquet file

- Run the CT process
    - Create sinewave data
    - Write the partitioned parquet CT sinewave data
    - Create peak amplitude data
    - Write the partitioned parquet CT peak amplitude data
        

In [14]:
parquet_cols_to_use = ['Id', 'ELR', 'TrackID', 'Location_m', 'Top_Right35', 'Top_Left35']   # Columns required for CT process

# Write the filenames to parquet_file_list[]
parquet_file_list = []                                      
for (dirpath, dirnames, filenames) in walk(data_in_path):
    parquet_file_list.extend(filenames)
    num_of_files = len(parquet_file_list)

for parquet_file in parquet_file_list:

    tg_parquet = pq.read_pandas(data_in_path + parquet_file).to_pandas()    # Read the old style parquet data
    
    fdate = ExtractDate(parquet_file)                                       # Get the file date from filename
    tg_parquet['Date']=datetime.strptime(fdate, "%Y-%m-%d").date()

    tg_parquet['Ref']=parquet_file[:-18]                                    # Remove all extension parts


    #------------------------------------------------------------------------------------------------------------------------------
    if BY_FILENAME:
        tg_parquet.to_parquet(
            data_out_tg_ref,
            partition_cols=ct_part_cols_filename,
            engine=engine,
            compression=comp,
            index=False
            )

    else:
        # Write the df (a single TG parquet file) to a partioned parquet file, for easy retrival later 
        tg_parquet.to_parquet(
            data_out_tg, 
            partition_cols=tg_part_cols, 
            engine=engine, 
            compression=comp
            )

    
    #___________________________________________________ Cyclic Top Processing - Sinewave________________________________________________________

    Cyclic_Tops = tg_parquet[parquet_cols_to_use]
    Cyclic_Tops = Cyclic_Tops.rename(columns={'Id':'CT_Output_Index'})      # Keeping this column in case record order gets mixed - not used anywhere
    Cyclic_Tops['Ref'] = tg_parquet['Ref']                                  # Filename to Ref column
    Cyclic_Tops = Cyclic_Tops[['CT_Output_Index', 'ELR', 'TrackID', 'Location_m', 'Top_Left35', 'Top_Right35', 'Ref']]

    sinewave = runfiles(Cyclic_Tops, fdate)                                 # Process the CT Sinewave output (pass the date string for output and partitioning)

    #________________________________________________ Cyclic Top Processing - Peak Amplitude_______________________________________________________

    output = generate_all_data_peaks(sinewave)             
    output = find_max_peaks(output)                        
    output = save_file_All_Data(output, fdate)                              # Output the Peak Amplitude File(s)

    num_of_files = num_of_files - 1 # type: ignore
    print(num_of_files, parquet_file)    


In [15]:
# Time taken in minutes
end_time = time.time()

print("Time to run (minutes) {:0.2f}\n".format((end_time - start_time)/60))
print("Time to run (Seconds) {:0.0f}\n".format(end_time - start_time))


Time to run (minutes) 0.00

Time to run (Seconds) 0

